# Zomato Data Analysis

Exploratory analysis on a Zomato restaurant dataset — covers rating distribution, vote counts by category, online ordering patterns, and cost breakdowns.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Load Dataset

In [ ]:
df = pd.read_csv('Zomato data .csv')
print(df.shape)
df.head()


## 3. Clean the `rate` Column

Ratings come in the format `4.1/5`. We strip the denominator and convert to float. Non-numeric entries like `'NEW'` or `'-'` are coerced to `NaN`.

In [ ]:
def parse_rate(val):
    try:
        return float(str(val).split('/')[0])
    except ValueError:
        return np.nan

df['rate'] = df['rate'].apply(parse_rate)
df['rate'].describe()


## 4. Clean the `approx_cost` Column

Cost values may contain commas (e.g. `1,200`). Strip them before converting to numeric.

In [ ]:
df['approx_cost(for two people)'] = (
    df['approx_cost(for two people)']
    .astype(str)
    .str.replace(',', '', regex=False)
)
df['approx_cost(for two people)'] = pd.to_numeric(
    df['approx_cost(for two people)'], errors='coerce'
)


## 5. Dataset Overview

In [ ]:
df.info()


## 6. Restaurant Type Distribution

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(x=df['listed_in(type)'])
plt.xlabel('Type of Restaurant')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7. Total Votes by Restaurant Type

In [ ]:
votes_by_type = df.groupby('listed_in(type)')['votes'].sum()

plt.figure(figsize=(10, 5))
plt.plot(votes_by_type.index, votes_by_type.values, color='green', marker='o')
plt.xlabel('Type of Restaurant', color='red', fontsize=14)
plt.ylabel('Total Votes', color='red', fontsize=14)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 8. Restaurant with the Most Votes

In [ ]:
top_idx = df['votes'].idxmax()
top_restaurant = df.loc[top_idx, 'name']
top_votes = df.loc[top_idx, 'votes']
print(f'Most voted restaurant: {top_restaurant} ({top_votes} votes)')


## 9. Online Order Availability

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x=df['online_order'])
plt.xlabel('Accepts Online Orders')
plt.tight_layout()
plt.show()


## 10. Rating Distribution

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df['rate'].dropna(), bins=5, edgecolor='black')
plt.title('Ratings Distribution')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


## 11. Approximate Cost for Two People

In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(df['approx_cost(for two people)'].dropna(), bins=20, kde=True)
plt.xlabel('Approx Cost for Two (₹)')
plt.title('Cost Distribution')
plt.tight_layout()
plt.show()


## 12. Online Order vs Rating (Box Plot)

In [ ]:
plt.figure(figsize=(6, 5))
sns.boxplot(x='online_order', y='rate', data=df)
plt.xlabel('Accepts Online Orders')
plt.ylabel('Rating')
plt.title('Rating by Online Order Availability')
plt.tight_layout()
plt.show()


## 13. Heatmap — Restaurant Type vs Online Order

In [ ]:
pivot = df.pivot_table(
    index='listed_in(type)',
    columns='online_order',
    aggfunc='size',
    fill_value=0
)

plt.figure(figsize=(8, 6))
sns.heatmap(pivot, annot=True, cmap='YlGnBu', fmt='d')
plt.title('Restaurant Type vs Online Order')
plt.xlabel('Online Order')
plt.ylabel('Restaurant Type')
plt.tight_layout()
plt.show()
